<a href="https://colab.research.google.com/github/rania-532/projet-deep-learning-4IAD/blob/main/deep-learning-4IAD.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# Monter Google Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
import os
print(os.listdir('/content/drive'))

['.shortcut-targets-by-id', 'MyDrive', '.Trash-0', '.Encrypted']


## Partie I : MLP et Ingénierie PyTorch  ### Qu'est-ce qu'un MLP ? Un Perceptron Multicouche (MLP) est le réseau de neurones le plus fondamental. Il est composé de couches entièrement connectées : chaque neurone est relié à tous les neurones de la couche suivante.  **Pourquoi un MLP pour les données tabulaires ?** Les données tabulaires n'ont pas de structure spatiale (pas d'images) ni de séquence temporelle. Un MLP est parfait car il peut apprendre des relations non-linéaires entre des variables indépendantes.  **Concepts clés :** - `nn.Module` : classe de base de PyTorch pour tous les modèles - Propagation avant : le signal va de l'entrée vers la sortie - Rétropropagation : le gradient va de la sortie vers l'entrée - `state_dict()` : dictionnaire contenant tous les paramètres du modèle

In [5]:
# === CELLULE 2 : Imports ===
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score,
                             recall_score, f1_score, confusion_matrix)

# Vérification GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device utilisé : {device}')


Device utilisé : cuda


In [6]:
# === CELLULE 3 : Préparation des données ===

# 1. Chargement — Wine Quality depuis sklearn
wine = load_wine()
X, y = wine.data, wine.target
print(f'Forme des données : {X.shape}')  # (178, 13)
print(f'Classes : {wine.target_names}')
print(f'Variables : {wine.feature_names}')

# 2. Analyse exploratoire
df = pd.DataFrame(X, columns=wine.feature_names)
df['target'] = y
print(df.describe())
print('Distribution des classes:', df['target'].value_counts())

# 3. Normalisation — OBLIGATOIRE pour les MLP
# Pourquoi ? Les variables ont des échelles très différentes.
# Ex: alcool (9-15) vs acide malique (0-5). Sans normalisation,
# le gradient favorise les grandes variables.
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 4. Séparation train/val/test (70/15/15)
X_train, X_temp, y_train, y_temp = train_test_split(
    X_scaled, y, test_size=0.30, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp)

print(f'Train: {X_train.shape[0]} | Val: {X_val.shape[0]} | Test: {X_test.shape[0]}')

# 5. Conversion en tenseurs PyTorch
def to_tensor(X, y):
    return (torch.FloatTensor(X).to(device),
            torch.LongTensor(y).to(device))

X_tr, y_tr = to_tensor(X_train, y_train)
X_v,  y_v  = to_tensor(X_val,   y_val)
X_te, y_te = to_tensor(X_test,  y_test)

# 6. DataLoaders
train_loader = DataLoader(TensorDataset(X_tr, y_tr), batch_size=32, shuffle=True)
val_loader   = DataLoader(TensorDataset(X_v,  y_v),  batch_size=32)


Forme des données : (178, 13)
Classes : ['class_0' 'class_1' 'class_2']
Variables : ['alcohol', 'malic_acid', 'ash', 'alcalinity_of_ash', 'magnesium', 'total_phenols', 'flavanoids', 'nonflavanoid_phenols', 'proanthocyanins', 'color_intensity', 'hue', 'od280/od315_of_diluted_wines', 'proline']
          alcohol  malic_acid         ash  alcalinity_of_ash   magnesium  \
count  178.000000  178.000000  178.000000         178.000000  178.000000   
mean    13.000618    2.336348    2.366517          19.494944   99.741573   
std      0.811827    1.117146    0.274344           3.339564   14.282484   
min     11.030000    0.740000    1.360000          10.600000   70.000000   
25%     12.362500    1.602500    2.210000          17.200000   88.000000   
50%     13.050000    1.865000    2.360000          19.500000   98.000000   
75%     13.677500    3.082500    2.557500          21.500000  107.000000   
max     14.830000    5.800000    3.230000          30.000000  162.000000   

       total_phenols 

In [7]:
# === CELLULE 4 : Les deux versions du MLP ===

# ----- Version 1 : nn.Sequential -----
# Avantage : concis, lisible pour des architectures simples
# Inconvénient : pas de flexibilité (pas de skip connections, etc.)

mlp_sequential = nn.Sequential(
    nn.Linear(13, 64),   # 13 entrées (features Wine)
    nn.ReLU(),
    nn.Dropout(0.3),     # Régularisation : évite le surapprentissage
    nn.Linear(64, 32),
    nn.ReLU(),
    nn.Dropout(0.2),
    nn.Linear(32, 3)     # 3 classes de sortie
).to(device)

# ----- Version 2 : Classe personnalisée -----
# Avantage : contrôle total, on peut ajouter des méthodes custom
# C'est la manière professionnelle d'implémenter un réseau

class MLP(nn.Module):
    def __init__(self, input_dim, hidden_dims, output_dim, dropout=0.3):
        super(MLP, self).__init__()
        # Construction dynamique des couches
        layers = []
        prev_dim = input_dim
        for dim in hidden_dims:
            layers += [nn.Linear(prev_dim, dim), nn.ReLU(), nn.Dropout(dropout)]
            prev_dim = dim
        layers.append(nn.Linear(prev_dim, output_dim))
        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x)

mlp_custom = MLP(input_dim=13, hidden_dims=[64, 32], output_dim=3).to(device)
print('Architecture MLP custom:')
print(mlp_custom)
print(f'Nombre de paramètres: {sum(p.numel() for p in mlp_custom.parameters()):,}')


Architecture MLP custom:
MLP(
  (network): Sequential(
    (0): Linear(in_features=13, out_features=64, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=64, out_features=32, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.3, inplace=False)
    (6): Linear(in_features=32, out_features=3, bias=True)
  )
)
Nombre de paramètres: 3,075


In [8]:
# === CELLULE 5 : Inspection avec named_parameters() et state_dict() ===

print('=== named_parameters() ===')
for name, param in mlp_custom.named_parameters():
    print(f'{name:40s} | shape: {str(param.shape):20s} | trainable: {param.requires_grad}')

print()
print('=== state_dict() — premières valeurs ===')
for key, val in mlp_custom.state_dict().items():
    print(f'{key:40s} -> {val.shape}')


=== named_parameters() ===
network.0.weight                         | shape: torch.Size([64, 13]) | trainable: True
network.0.bias                           | shape: torch.Size([64])     | trainable: True
network.3.weight                         | shape: torch.Size([32, 64]) | trainable: True
network.3.bias                           | shape: torch.Size([32])     | trainable: True
network.6.weight                         | shape: torch.Size([3, 32])  | trainable: True
network.6.bias                           | shape: torch.Size([3])      | trainable: True

=== state_dict() — premières valeurs ===
network.0.weight                         -> torch.Size([64, 13])
network.0.bias                           -> torch.Size([64])
network.3.weight                         -> torch.Size([32, 64])
network.3.bias                           -> torch.Size([32])
network.6.weight                         -> torch.Size([3, 32])
network.6.bias                           -> torch.Size([3])


In [9]:
# === CELLULE 6 : Trois stratégies d'initialisation ===

def init_weights(model, strategy='xavier'):
    for m in model.modules():
        if isinstance(m, nn.Linear):
            if strategy == 'gaussian':
                nn.init.normal_(m.weight, mean=0, std=0.01)
                nn.init.zeros_(m.bias)
            elif strategy == 'constant':
                nn.init.constant_(m.weight, 0.01)
                nn.init.zeros_(m.bias)
            elif strategy == 'xavier':
                nn.init.xavier_uniform_(m.weight)  # Meilleure pratique
                nn.init.zeros_(m.bias)

# Test des trois stratégies
results_init = {}
for strategy in ['gaussian', 'constant', 'xavier']:
    model = MLP(input_dim=13, hidden_dims=[64,32], output_dim=3).to(device)
    init_weights(model, strategy)
    # Entraînement rapide (5 epochs pour comparer)
    opt = optim.Adam(model.parameters(), lr=1e-3)
    loss_fn = nn.CrossEntropyLoss()
    losses = []
    for epoch in range(5):
        model.train()
        for xb, yb in train_loader:
            opt.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            opt.step()
            losses.append(loss.item())
    results_init[strategy] = np.mean(losses[-10:])
    print(f'{strategy:12s} | loss finale (moy): {results_init[strategy]:.4f}')


gaussian     | loss finale (moy): 1.0893
constant     | loss finale (moy): 1.0567
xavier       | loss finale (moy): 0.9557


In [10]:
# === CELLULE 7 : Entraînement avec le meilleur modèle ===

model = MLP(input_dim=13, hidden_dims=[64, 32], output_dim=3).to(device)
init_weights(model, 'xavier')
optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss()
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=10)

train_losses, val_losses, val_accs = [], [], []
best_val_acc = 0
EPOCHS = 100

for epoch in range(EPOCHS):
    # --- Phase entraînement ---
    model.train()
    epoch_loss = 0
    for xb, yb in train_loader:
        optimizer.zero_grad()
        preds = model(xb)
        loss = criterion(preds, yb)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    train_losses.append(epoch_loss / len(train_loader))

    # --- Phase validation ---
    model.eval()
    with torch.no_grad():
        val_preds = model(X_v)
        val_loss = criterion(val_preds, y_v).item()
        val_acc = (val_preds.argmax(1) == y_v).float().mean().item()
    val_losses.append(val_loss)
    val_accs.append(val_acc)
    scheduler.step(val_loss)

    # --- Sauvegarde du meilleur modèle ---
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), '/content/drive/MyDrive/best_mlp.pth')

    if (epoch+1) % 20 == 0:
        print(f'Epoch {epoch+1:3d} | Loss train: {train_losses[-1]:.4f} | Loss val: {val_loss:.4f} | Acc val: {val_acc:.4f}')


Epoch  20 | Loss train: 0.2454 | Loss val: 0.1496 | Acc val: 1.0000
Epoch  40 | Loss train: 0.0972 | Loss val: 0.0451 | Acc val: 1.0000
Epoch  60 | Loss train: 0.0308 | Loss val: 0.0231 | Acc val: 1.0000
Epoch  80 | Loss train: 0.0325 | Loss val: 0.0154 | Acc val: 1.0000
Epoch 100 | Loss train: 0.0086 | Loss val: 0.0138 | Acc val: 1.0000
